# 07: compare the EN and JA arms (local, CPU)

In [ ]:
N_BOOT = 10000


In [ ]:
# 07_compare (local CPU): EN adapter vs JA adapter on the same prompts.
import glob, hashlib, json, os, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest

MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
PT3 = MYDRIVE / "ez-penta" / "post_train_3"
OUT = PT3 / "compare"
OUT.mkdir(parents=True, exist_ok=True)
A = {a: PT3 / f"arm_{a}" for a in ("en", "ja")}
for a in A:
    assert (A[a] / "DONE").exists(), f"arm_{a}/DONE missing"
frozen = {a: json.loads((A[a] / "frozen_pool.json").read_text()) for a in A}
checks = {}
same_frozen = frozen["en"]["prompt_ids_sha256"] == frozen["ja"]["prompt_ids_sha256"]
pids = set(frozen["en"]["prompt_ids"]) & set(frozen["ja"]["prompt_ids"])
checks["frozen_sets"] = {"identical": same_frozen, "en": frozen["en"]["n_prompts"], "ja": frozen["ja"]["n_prompts"], "intersection": len(pids)}
tid = {a: json.loads((A[a] / "train" / "train_item_ids.json").read_text()) for a in A}
th = {a: (A[a] / "train" / "train_items_sha256.txt").read_text().strip() for a in A}
inter = set(tid["en"]) & set(tid["ja"])
checks["training_items"] = {"hash_en": th["en"], "hash_ja": th["ja"], "identical": th["en"] == th["ja"],
                            "n_en": len(tid["en"]), "n_ja": len(tid["ja"]), "shared": len(inter),
                            "overlap_pct": round(100 * len(inter) / max(len(set(tid["en"]) | set(tid["ja"])), 1), 1)}
am = {a: json.loads((A[a] / "adapters" / "manifest.json").read_text()) for a in A}
hw = {a: (am[a].get("hardware"), am[a].get("dtype")) for a in A}
checks["hardware"] = {"en": hw["en"], "ja": hw["ja"], "matched": hw["en"] == hw["ja"]}
HW_TAG = "" if hw["en"] == hw["ja"] else " (hardware-mismatched)"
print(json.dumps(checks, indent=1))

# Prompt metadata from the shared batches (identical for both arms).
batches = sorted(set(frozen["en"]["batches"]) & set(frozen["ja"]["batches"]))
meta = pd.concat([pd.read_json(PT3 / "eval_pool" / "batches" / f"b{n:04d}" / "prompts.jsonl", lines=True) for n in batches])
# Resolve each frozen seed to the first batch containing it (a recurring seed ID's later copy is not in the pool).
fseeds = set(frozen["en"]["seed_ids"]) & set(frozen["ja"]["seed_ids"])
first_batch = meta[meta.seed_id.isin(fseeds)].groupby("seed_id").batch.min()
keys = set(zip(first_batch.values, first_batch.index))
meta = meta[[(b, s) in keys for b, s in zip(meta.batch, meta.seed_id)]]
assert meta.prompt_id.is_unique
pids = set(meta.prompt_id) & pids
checks["frozen_sets"]["unique_prompts_analysed"] = len(pids)
meta = meta[meta.prompt_id.isin(pids)].set_index("prompt_id")[["batch", "seed_id", "category", "framing", "lang", "kind", "severity"]]

_JA = re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LA = re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(t):
    j, l = len(_JA.findall(t or "")), len(_LA.findall(t or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


def load_arm(a):
    fr = []
    for n in batches:
        d = pd.read_json(A[a] / "eval" / f"b{n:04d}.jsonl", lines=True)
        fr.append(d[["model", "prompt_id", "answer", "finish_reason", "g06_label", "g06_refused", "g4b_label", "g4b_refused"]])
    d = pd.concat(fr, keys=batches, names=["_b", None]).reset_index(level=0)
    d = d[d.prompt_id.isin(pids)]
    return d[d._b.values == meta.loc[d.prompt_id, "batch"].values].drop(columns="_b")


E = {a: load_arm(a) for a in A}
seed_models = {a: sorted(m for m in E[a].model.unique() if re.search(r"_s\d$", m)) for a in A}
print("adapter models:", seed_models)
assert len(seed_models["en"]) == len(seed_models["ja"]) >= 1
BASE_ARM = "en" if hw["en"] == hw["ja"] else "en"
rows = [E[BASE_ARM][E[BASE_ARM].model == "base"].assign(arm="base", run=-1)]
for a in A:
    for r, m in enumerate(seed_models[a]):
        rows.append(E[a][E[a].model == m].assign(arm=a, run=r))
D = pd.concat(rows, ignore_index=True)
D["answer_lang"] = D.answer.map(answer_lang)
D["chars"] = D.answer.str.len()
for j in ("g06", "g4b"):
    D[f"{j}_unsafe_sc"] = D[f"{j}_label"].map({"Unsafe": 1.0, "Safe": 0.0})
    D[f"{j}_notunsafe"] = (D[f"{j}_label"] != "Unsafe").astype(float)
    D[f"{j}_ref"] = D[f"{j}_refused"].astype(float)
D = D.join(meta.drop(columns="batch"), on="prompt_id")
RUNS = len(seed_models["en"])
print(len(D), "rows;", RUNS, "runs per arm")


In [ ]:
# C1-C4: EN adapter vs JA adapter (per-run McNemar, two-level bootstrap), Holm over C1-C3.
rng = np.random.default_rng(0)


def piv(col):
    return D.pivot_table(index="prompt_id", columns=["arm", "run"], values=col, aggfunc="first")


def base_fail(j):
    b = D[(D.arm == "base")].set_index("prompt_id")
    return set(b.index[(b.kind == "harmful") & (b.severity == "clear") & (b[f"{j}_unsafe_sc"] == 1)])


def boot_diff(pr, lang_sign, col_kind):
    """pr: frame indexed by prompt with columns (arm, run) and seed_id, lang. Statistic: mean over runs of
    sum(lang_sign * (EN - JA)) / n, with eval-seed cluster bootstrap and run resampling (EN run r paired with JA run r)."""
    seeds_u = sorted(pr.seed_id.unique())
    idx = {s: i for i, s in enumerate(seeds_u)}
    ci = pr.seed_id.map(idx).values
    S = len(seeds_u)
    num, den = np.zeros((RUNS, S)), np.zeros((RUNS, S))
    for r in range(RUNS):
        e, q = pr[("en", r)].values, pr[("ja", r)].values
        ok = ~np.isnan(e) & ~np.isnan(q)
        np.add.at(num[r], ci[ok], (lang_sign * (e - q))[ok])
        np.add.at(den[r], ci[ok], 1)
    obs = float(np.mean([num[r].sum() / den[r].sum() for r in range(RUNS) if den[r].sum()]))
    bs = np.empty(N_BOOT)
    for b in range(N_BOOT):
        w = np.bincount(rng.integers(0, S, S), minlength=S)
        rr = rng.integers(0, RUNS, RUNS)
        v = [(num[r] @ w) / (den[r] @ w) for r in rr if den[r] @ w > 0]
        bs[b] = np.mean(v) if v else np.nan
    p = float(min(1.0, 2 * min(np.nanmean(bs <= 0), np.nanmean(bs >= 0))))
    lo, hi = np.nanpercentile(bs, [2.5, 97.5])
    return obs, [float(lo), float(hi)], p


def mcnemar(pr):
    out = {}
    for r in range(RUNS):
        s = pr[[("en", r), ("ja", r)]].dropna()
        e1 = int(((s[("en", r)] == 1) & (s[("ja", r)] == 0)).sum())
        q1 = int(((s[("en", r)] == 0) & (s[("ja", r)] == 1)).sum())
        out[f"run{r}"] = {"n": int(len(s)), "en_only": e1, "ja_only": q1, "p": float(binomtest(e1, e1 + q1).pvalue) if e1 + q1 else 1.0}
    return out


def frame(col, mask):
    p = piv(col)
    p = p[[c for c in p.columns if c[0] in ("en", "ja")]]
    m = meta.loc[p.index]
    sel = mask(m)
    p = p[sel.values]
    p["seed_id"] = m.seed_id[sel.values].values
    p["lang"] = m.lang[sel.values].values
    return p


def run_tests(j):
    bf = base_fail(j)
    res = {}
    for cid, lang in (("C1", "ja"), ("C2", "en")):
        pr = frame(f"{j}_notunsafe", lambda m: m.index.isin(bf) & (m.lang == lang))
        o, ci, p = boot_diff(pr, 1.0, "fix")
        res[cid] = {"what": f"fix rate, {lang} baseline failures (EN - JA)", "n_prompts": len(pr), "diff": o, "ci95": ci, "p": p, "mcnemar": mcnemar(pr)}
    for cid, mask, what in (("C3", lambda m: (m.kind == "twin") & m.framing.isin(["control", "fc"]), "over-refusal interaction, twins control+fc"),
                            ("C4", lambda m: (m.kind == "harmful") & (m.severity == "borderline"), "over-caution interaction, borderline")):
        pr = frame(f"{j}_ref", mask)
        sign = np.where(pr.lang.values == "en", 1.0, -1.0)
        # interaction = (EN-JA on en prompts) - (EN-JA on ja prompts), estimated as two separate rates
        en_part, ja_part = pr[pr.lang == "en"], pr[pr.lang == "ja"]
        o1, _, _ = boot_diff(en_part, 1.0, "x")
        o2, _, _ = boot_diff(ja_part, 1.0, "x")
        # bootstrap the interaction jointly over eval seeds (each seed has both languages)
        seeds_u = sorted(pr.seed_id.unique()); idx = {s: i for i, s in enumerate(seeds_u)}; S = len(seeds_u)
        ci_ = pr.seed_id.map(idx).values
        num = np.zeros((2, RUNS, S)); den = np.zeros((2, RUNS, S))
        for li, lg in enumerate(("en", "ja")):
            lm = pr.lang.values == lg
            for r in range(RUNS):
                e, q = pr[("en", r)].values, pr[("ja", r)].values
                ok = lm & ~np.isnan(e) & ~np.isnan(q)
                np.add.at(num[li, r], ci_[ok], (e - q)[ok]); np.add.at(den[li, r], ci_[ok], 1)
        bs = np.empty(N_BOOT)
        for b in range(N_BOOT):
            w = np.bincount(rng.integers(0, S, S), minlength=S); rr = rng.integers(0, RUNS, RUNS)
            bs[b] = np.mean([(num[0, r] @ w) / max(den[0, r] @ w, 1) - (num[1, r] @ w) / max(den[1, r] @ w, 1) for r in rr])
        p = float(min(1.0, 2 * min(np.mean(bs <= 0), np.mean(bs >= 0))))
        res[cid] = {"what": what + " (EN-JA on en prompts) - (EN-JA on ja prompts)", "n_prompts": len(pr),
                    "en_prompts_diff": o1, "ja_prompts_diff": o2, "diff": o1 - o2,
                    "ci95": [float(x) for x in np.percentile(bs, [2.5, 97.5])], "p": p,
                    "mcnemar_en_prompts": mcnemar(en_part), "mcnemar_ja_prompts": mcnemar(ja_part)}
    # Holm over C1-C3
    ps = sorted(((res[c]["p"], c) for c in ("C1", "C2", "C3")))
    adj, run_max = {}, 0.0
    for i, (p, c) in enumerate(ps):
        run_max = max(run_max, min(1.0, (3 - i) * p))
        adj[c] = run_max
    for c in adj:
        res[c]["p_holm"] = adj[c]
    return res


T = {j: run_tests(j) for j in ("g06", "g4b")}
for c in ("C1", "C2", "C3"):
    a, b = T["g06"][c], T["g4b"][c]
    T["g06"][c]["confirmed"] = bool(np.sign(a["diff"]) == np.sign(b["diff"]) and a["p_holm"] < 0.05 and b["p_holm"] < 0.05)
for j in T:
    for c, r in T[j].items():
        print(j, c, f"diff {r['diff']:+.3f} CI {r['ci95']} p {r['p']:.3g}", f"holm {r.get('p_holm', float('nan')):.3g}")


In [ ]:
# Descriptive (no tests): framing, category, wrong-language rate, answer length, by arm x prompt language.
def rate(g, col):
    v = g[col].dropna()
    return f"{int(v.sum())}/{len(v)} ({v.mean():.1%})" if len(v) else "-"


desc = {}
D["wrong_lang"] = (D.answer_lang != D.lang).astype(float)
harm = D[D.kind == "harmful"]
twin = D[(D.kind == "twin") & D.framing.isin(["control", "fc"])]
for name, sub, col, by in (("unsafe_by_framing", harm, "g06_unsafe_sc", "framing"), ("unsafe_by_category", harm, "g06_unsafe_sc", "category"),
                           ("overrefusal_by_framing", D[D.kind == "twin"], "g06_ref", "framing"), ("wrong_language", D, "wrong_lang", None)):
    keys = ["arm", "lang"] + ([by] if by else [])
    t = sub.groupby(keys).apply(lambda g: rate(g, col)).unstack("arm") if by else sub.groupby(keys).apply(lambda g: rate(g, col)).unstack("arm")
    desc[name] = t
desc["mean_answer_chars"] = D.groupby(["arm", "lang", "kind"]).chars.mean().round(0).unstack("arm")


In [ ]:
# results.json, report.md (with the combined error summary of both arms), DONE.
results = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "checks": checks, "base_from": BASE_ARM,
           "tests_g06": T["g06"], "tests_g4b": T["g4b"], "descriptive": {k: v.reset_index().to_dict(orient="records") for k, v in desc.items()}}
(OUT / "results.json").write_text(json.dumps(results, indent=1, default=str))


def tline(c, r):
    s = f"| {c}{HW_TAG} | {r['what']} | {r['n_prompts']} | {r['diff']:+.3f} | [{r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f}] | {r['p']:.3g} | "
    return s + (f"{r['p_holm']:.3g}" if "p_holm" in r else "–") + " |\n"


rep = ["# post_train_3: EN-trained vs JA-trained layer-18 LoRA\n", "## Checks\n",
       f"- **Frozen sets:** identical = {checks['frozen_sets']['identical']}; en {checks['frozen_sets']['en']}, ja {checks['frozen_sets']['ja']} prompts; analysed intersection {checks['frozen_sets']['intersection']}.",
       f"- **Training items:** identical hash = {checks['training_items']['identical']}; shared {checks['training_items']['shared']} item IDs ({checks['training_items']['overlap_pct']}% of the union).",
       f"- **Hardware and dtype:** en {checks['hardware']['en']}, ja {checks['hardware']['ja']}; matched = {checks['hardware']['matched']}.",
       f"- Base answers from the {BASE_ARM} arm.\n"]
for j, title in (("g06", "Primary judge (Qwen3Guard-Gen-0.6B)"), ("g4b", "Second judge (Qwen3Guard-Gen-4B)")):
    rep += [f"## Tests: {title}\n", "Diff = EN adapter − JA adapter (mean over paired training runs). C3/C4 are interactions. Holm over C1–C3.\n",
            "| ID | what | n | diff | 95% CI | p | p (Holm) |\n|---|---|---|---|---|---|---|\n" + "".join(tline(c, r) for c, r in T[j].items())]
rep += ["Confirmed (both judges same direction, Holm p < 0.05): " + ", ".join(f"{c}: {T['g06'][c]['confirmed']}" for c in ("C1", "C2", "C3")) + "\n"]
rep += ["## Descriptive\n"]
for k, v in desc.items():
    rep += [f"### {k}\n", f"{k.replace('_', ' ')}, by arm and prompt language.\n", v.to_markdown() + "\n"]
rep += ["## Combined error summary (both arms)\n"]
for a in A:
    for f in ("ERROR_LOG.md", "DEVIATIONS.md"):
        p = A[a] / f
        rep += [f"### arm_{a}/{f}\n", p.read_text() if p.exists() else "(missing)\n"]
(OUT / "report.md").write_text("\n".join(rep))
(OUT / "DONE").write_text(hashlib.sha256((OUT / "results.json").read_bytes()).hexdigest() + "\n")
print("wrote compare/results.json, report.md, DONE")
